# Type-B Stage 2 — Loss Function Comparison

**Hypothesis:** CombinedLoss (MSE + Cosine) outperforms MSELoss across all CNN architectures by directly aligning the training objective with the cosine-based retrieval metric.

**Fixed:** `tinybert_mean` (312-dim) — best embedding from Stage 1  
**Target normalisation:**
- `mse` → raw targets (L2 distance in original embedding space)
- `combined` → L2-normalised targets (balances MSE and Cosine loss scales on the unit sphere)

**Models:** `cnn_1layer`, `cnn_3layer`, `resnet18_pt`  
**Losses:** `mse`, `combined`  
**Total runs:** 3 × 2 = 6

| Run | Model | Loss | Target norm | Epochs |
|---|---|---|---|---|
| b_s2_cnn_1layer_mse_tinybert_mean | cnn_1layer | MSELoss | raw | 30 |
| b_s2_cnn_1layer_combined_tinybert_mean_normed | cnn_1layer | CombinedLoss | L2-normed | 30 |
| b_s2_cnn_3layer_mse_tinybert_mean | cnn_3layer | MSELoss | raw | 30 |
| b_s2_cnn_3layer_combined_tinybert_mean_normed | cnn_3layer | CombinedLoss | L2-normed | 30 |
| b_s2_resnet18_pt_mse_tinybert_mean | resnet18_pt | MSELoss | raw | 20 |
| b_s2_resnet18_pt_combined_tinybert_mean_normed | resnet18_pt | CombinedLoss | L2-normed | 20 |

## Cell 1 — Config

In [ ]:
# ── MODE ──────────────────────────────────────────────────────────────────────
MODE = 'colab'   # 'local' or 'colab'

# ── Local config ──────────────────────────────────────────────────────────────
LOCAL_REPO_DIR = '/Users/yeon/work/UoB_CourseWork/uob-ds-intro-to-ai-final-cw-2026'

# ── Colab config ──────────────────────────────────────────────────────────────
GITHUB_REPO_URL = 'https://github.com/Rylie-KIM/uob-ds-intro-to-ai-final-cw-2026'
DRIVE_BASE      = '/content/drive/MyDrive/uob-ds-ai'

# ── Experiment config ─────────────────────────────────────────────────────────
# Re-running MSELoss only — previous runs used normalised targets (incorrect for MSE).
# CombinedLoss runs are correct as-is and do NOT need to be re-run.
MODELS = ['cnn_1layer', 'cnn_3layer', 'resnet18_pt']
LOSSES = ['mse']

# Per-model epoch override (None = use train_type_b_stage2.py defaults)
EPOCH_OVERRIDE: dict | None = None
# Example override: EPOCH_OVERRIDE = {'cnn_1layer': 20, 'cnn_3layer': 25, 'resnet18_pt': 15}

# ── Run tag ──────────────────────────────────────────────────────────────────
import datetime as _dt
RUN_TAG = f'stage2_{_dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")}'
# ─────────────────────────────────────────────────────────────────────────────

print(f'MODE    = {MODE}')
print(f'RUN_TAG = {RUN_TAG!r}')
print(f'MODELS  = {MODELS}')
print(f'LOSSES  = {LOSSES}')
print(f'Total runs: {len(MODELS) * len(LOSSES)}')

## Cell 2-1 — Check corrupted images (Colab only)

In [ ]:
from pathlib import Path
from PIL import Image

img_dir = Path('/content/images/type-b')
if img_dir.exists():
    corrupted = []
    for f in sorted(img_dir.glob('*.png')):
        try:
            Image.open(f).verify()
        except Exception:
            corrupted.append(f)
    print(f'Corrupted: {len(corrupted)} files')
    for f in corrupted:
        print(f'  deleting {f.name}')
        f.unlink()
    print('Done.')
else:
    print('Image dir not found — will be created in Cell 2-2.')

## Cell 2-2 — Setup

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path

if MODE == 'local':
    REPO_DIR = LOCAL_REPO_DIR
    print(f'Local mode. Repo: {REPO_DIR}')

elif MODE == 'colab':
    from google.colab import drive, userdata
    drive.mount('/content/drive')

    REPO_DIR      = '/content/repo'
    DRIVE_DATA    = f'{DRIVE_BASE}/data'
    DRIVE_RESULTS = f'{DRIVE_BASE}/results'

    try:
        token = userdata.get('GITHUB_TOKEN')
        repo_url_auth = GITHUB_REPO_URL.replace('https://', f'https://{token}@')
    except Exception:
        print('WARNING: GITHUB_TOKEN not found in Colab Secrets.')
        repo_url_auth = GITHUB_REPO_URL

    if not os.path.exists(REPO_DIR):
        print('Cloning repo...')
        subprocess.run(['git', 'clone', repo_url_auth, REPO_DIR], check=True)
    else:
        print('Repo exists. Pulling latest...')
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
        commit = subprocess.check_output(
            ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
        print(f'  HEAD: {commit}')

    # ── Copy train+val images (exclude test split) ────────────────────────────
    import pandas as _pd

    def _get_test_filenames(repo_dir: str) -> set:
        splits_csv    = Path(repo_dir) / 'src/data/type-b/splits/type_b_splits_seed42.csv'
        image_map_csv = Path(repo_dir) / 'src/data/type-b/image_map_b.csv'
        sentences_csv = Path(repo_dir) / 'src/data/type-b/sentences_b.csv'
        splits_df = _pd.read_csv(splits_csv)
        image_map = _pd.read_csv(image_map_csv)
        sentences = _pd.read_csv(sentences_csv)
        df        = image_map.merge(sentences, on='sentence_id')
        test_idx  = splits_df[splits_df['split'] == 'test']['idx'].tolist()
        return set(df.iloc[test_idx]['filename'].tolist())

    test_filenames = _get_test_filenames(REPO_DIR)
    print(f'[info] {len(test_filenames)} test images excluded')

    LOCAL_IMG_BASE = Path('/content/images')
    local_img = LOCAL_IMG_BASE / 'type-b'
    drive_img = Path(DRIVE_DATA) / 'images' / 'type-b'
    repo_img  = Path(REPO_DIR) / 'src/data/images/type-b'

    local_img.mkdir(parents=True, exist_ok=True)
    all_drive_files = sorted(drive_img.glob('*.png'))
    src_files   = [f for f in all_drive_files if f.name not in test_filenames]
    local_files = set(f.name for f in local_img.glob('*.png'))
    missing     = [f for f in src_files if f.name not in local_files]

    print(f'[info] {len(all_drive_files)} total | {len(src_files)} train+val | {len(missing)} to copy')
    if missing:
        t0 = time.time()
        for i, src in enumerate(missing, 1):
            shutil.copy2(str(src), str(local_img / src.name))
            if i % 500 == 0 or i == len(missing):
                elapsed = time.time() - t0
                eta     = elapsed / i * (len(missing) - i)
                print(f'  {i}/{len(missing)} ({i/len(missing)*100:.0f}%)  eta={eta:.0f}s', end='\r')
        print(f'\n[done] copied in {time.time()-t0:.1f}s')
    else:
        print('[skip] all train+val images already on disk')

    if repo_img.is_symlink(): os.unlink(str(repo_img))
    elif repo_img.exists():   shutil.rmtree(str(repo_img))
    os.symlink(str(local_img), str(repo_img))
    print(f'[symlinked] repo images → /content/images/type-b')

    # ── Restore existing results from Drive ───────────────────────────────────
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        drive_sub = Path(DRIVE_RESULTS) / sub
        repo_sub  = Path(REPO_DIR) / 'src/pipelines/results' / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        repo_sub.mkdir(parents=True, exist_ok=True)
        copied = sum(
            1 for f in drive_sub.glob('*')
            if f.is_file() and not (repo_sub / f.name).exists()
            and shutil.copy2(str(f), str(repo_sub / f.name)) is not None or True
        )
        print(f'[restored] {sub}')

    # ── Copy tinybert_mean .pt from Drive ─────────────────────────────────────
    drive_emb = Path(DRIVE_RESULTS) / 'embeddings'
    repo_emb  = Path(REPO_DIR) / 'src/embeddings/computed-embeddings/type-b/results'
    drive_emb.mkdir(parents=True, exist_ok=True)
    repo_emb.mkdir(parents=True, exist_ok=True)
    pt_name  = 'tinybert_mean_embedding_result_typeb.pt'
    pt_repo  = repo_emb / pt_name
    pt_drive = drive_emb / pt_name
    if not pt_repo.exists():
        if pt_drive.exists():
            shutil.copy2(str(pt_drive), str(pt_repo))
            print(f'[copied] {pt_name}')
        else:
            print(f'WARNING: {pt_name} not found in Drive/embeddings/')
    else:
        print(f'[exists] {pt_name}')

    print(f'\nColab setup complete. Repo: {REPO_DIR}')

# sys.path
for p in [REPO_DIR, str(Path(REPO_DIR) / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## Cell 3 — Install Dependencies

In [ ]:
if MODE == 'colab':
    !pip install -q sentence-transformers gensim python-docx datasets
    print('Dependencies installed.')
else:
    print('Local mode — skipping pip install.')

## Cell 4 — Verify Paths

In [ ]:
checks = {
    'sentences_b.csv'        : Path(REPO_DIR) / 'src/data/type-b/sentences_b.csv',
    'image_map_b.csv'        : Path(REPO_DIR) / 'src/data/type-b/image_map_b.csv',
    'images/type-b/'         : Path(REPO_DIR) / 'src/data/images/type-b',
    'splits/'                : Path(REPO_DIR) / 'src/data/type-b/splits',
    'train_type_b_stage2.py' : Path(REPO_DIR) / 'src/pipelines/training/type-b/train_type_b_stage2.py',
    'cnn_1layer.py'          : Path(REPO_DIR) / 'src/models/type-b/cnn_1layer.py',
    'cnn_3layer.py'          : Path(REPO_DIR) / 'src/models/type-b/cnn_3layer.py',
    'resnet18_pt.py'         : Path(REPO_DIR) / 'src/models/type-b/resnet18_pt.py',
    'tinybert_mean.pt'       : Path(REPO_DIR) / 'src/embeddings/computed-embeddings/type-b/results/tinybert_mean_embedding_result_typeb.pt',
}
all_ok = True
for name, path in checks.items():
    ok = path.exists()
    print(f'  {"OK" if ok else "MISSING"}  {name:30s}  {path}')
    if not ok:
        all_ok = False
print()
print('All paths OK — ready to train.' if all_ok else 'Fix missing paths before continuing.')

## Cell 5 — Train (6 runs)

Runs all `(model, loss)` combinations sequentially.  
Results are backed up to Drive after **each run** so a Colab disconnect loses at most one run.

In [ ]:
import importlib.util as _ilu

# Evict stale cached config modules
for _key in list(sys.modules.keys()):
    if 'config.paths' in _key or _key in ('src.config', 'config'):
        del sys.modules[_key]

_spec = _ilu.spec_from_file_location(
    'train_type_b_stage2',
    Path(REPO_DIR) / 'src/pipelines/training/type-b/train_type_b_stage2.py',
)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

print(f'[paths] METRICS_B     = {_mod.METRICS_B}')
print(f'[paths] CHECKPOINTS_B = {_mod.CHECKPOINTS_B}')
assert str(_mod.METRICS_B).endswith('metrics/type-b'),      f'METRICS_B wrong: {_mod.METRICS_B}'
assert str(_mod.CHECKPOINTS_B).endswith('checkpoints/type-b'), f'CHECKPOINTS_B wrong: {_mod.CHECKPOINTS_B}'
print('[paths] OK\n')


def _backup_to_drive():
    if MODE != 'colab':
        return
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        repo_sub  = Path(REPO_DIR) / 'src/pipelines/results' / sub
        drive_sub = Path(DRIVE_RESULTS) / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        copied = 0
        for f in sorted(repo_sub.glob('*')):
            if f.is_file():
                shutil.copy2(str(f), str(drive_sub / f.name))
                copied += 1
        print(f'  [backup] {sub} ({copied} files)')


total = len(MODELS) * len(LOSSES)
for n, (model_name, loss_key) in enumerate(
    [(m, l) for m in MODELS for l in LOSSES], 1
):
    epochs = EPOCH_OVERRIDE.get(model_name) if EPOCH_OVERRIDE else None
    print(f'\n[{n}/{total}] {model_name} × {loss_key}  tag={RUN_TAG!r}')
    _mod.run_experiment_stage2(
        model_name=model_name,
        loss_key=loss_key,
        epochs=epochs,
        device=device,
        run_tag=RUN_TAG,
    )
    _backup_to_drive()

print('\nAll Stage 2 runs complete.')

## Cell 5.5 — (Optional) Manual re-backup

In [ ]:
if MODE == 'colab':
    for sub in ['checkpoints/type-b', 'metrics/type-b', 'figures/type-b']:
        repo_sub  = Path(REPO_DIR) / 'src/pipelines/results' / sub
        drive_sub = Path(DRIVE_RESULTS) / sub
        drive_sub.mkdir(parents=True, exist_ok=True)
        copied = 0
        for f in repo_sub.glob('*'):
            if f.is_file():
                shutil.copy2(str(f), str(drive_sub / f.name))
                copied += 1
        print(f'[backed up] {sub} → Drive  ({copied} files)')
else:
    print('Local mode — Drive backup skipped.')

## Cell 6 — Training Curves

In [ ]:
import pandas as pd

metrics_dir = Path(REPO_DIR) / 'src/pipelines/results/metrics/type-b'

for model_name in MODELS:
    for loss_key in LOSSES:
        use_norm = (loss_key == 'combined')
        run_name = (
            f'b_s2_{model_name}_{loss_key}_tinybert_mean_normed_{RUN_TAG}'
            if use_norm else
            f'b_s2_{model_name}_{loss_key}_tinybert_mean_{RUN_TAG}'
        )
        log_path = metrics_dir / f'{run_name}_training_log.csv'
        print(f'\n{"="*60}\n  {run_name}\n{"="*60}')
        if log_path.exists():
            log = pd.read_csv(log_path)
            print(f'[{len(log)} epochs]')
            display(log[['epoch', 'train_loss', 'val_loss', 'val_top1', 'val_mrr', 'val_median_rank', 'lr']])
        else:
            print(f'[missing] {log_path.name}')

## Cell 7 — Save CSV Results to Drive

In [ ]:
if MODE != 'colab':
    print('Local mode — Drive save skipped.')
else:
    import shutil
    from pathlib import Path

    metrics_src = Path(REPO_DIR) / 'src/pipelines/results/metrics/type-b'
    metrics_dst = Path(DRIVE_RESULTS) / 'metrics/type-b'
    metrics_dst.mkdir(parents=True, exist_ok=True)

    saved = []
    for model_name in MODELS:
        for loss_key in LOSSES:
            use_norm = (loss_key == 'combined')
            run_name = (
                f'b_s2_{model_name}_{loss_key}_tinybert_mean_normed_{RUN_TAG}'
                if use_norm else
                f'b_s2_{model_name}_{loss_key}_tinybert_mean_{RUN_TAG}'
            )
            csv_name = f'{run_name}_training_log.csv'
            src = metrics_src / csv_name
            dst = metrics_dst / csv_name
            if src.exists():
                shutil.copy2(str(src), str(dst))
                saved.append(csv_name)
                print(f'  [saved] {csv_name}')
            else:
                print(f'  [missing] {csv_name}')

    print(f'\nDone. {len(saved)} CSV(s) saved to Drive: {metrics_dst}')